# AeroSync-Mamba v31 — Final report: F1 (content-gated fusion) with Condition D as the comparison

**This is a reporting notebook, not an experiment.**
- **No model is trained or run.** `torch` is never imported, and a final hook checks that it was not loaded.
- **Sources:** everything comes from the saved day-out results of v29 (F1, D, A, I, F0, F2) and v30 (CA, CAG):
  per-run CSVs, pooled tables and per-window probability `.npz` files.
- **Checkpoints are not needed.** Every per-window quantity used here, including F1's gate values g_a / g_i, was saved
  at training time.

**Protocol for every number and figure:**
- leave-one-recording-day-out, 5 folds, with 2021-12-29 merged into 2021-12-31;
- 95 missions and 5238 paired 2.56 s windows (content sha1 `44c04ca18ec7a131`);
- seeds 42 and 123, probabilities averaged per window over the 2 seeds;
- mission prediction = argmax of the mission's mean window probability;
- 95 % CIs from a mission-level cluster bootstrap: 2000 resamples of missions within class, never windows.

**Models:**
- **F1, the headline model:** new encoders (an 80-weight sigmoid feature mask on the audio; one Conv1d k = 7 in front
  of the IMU CNN), then a **content-adaptive softmax gate** [g_a, g_i] computed from both modalities' summaries, then
  2 real Mamba blocks.
- **D, the comparison model, shown throughout:** the original encoders, then LACA's content-only bidirectional
  cross-attention (lag bias fixed at β = 0), then `[H_a ; SEP ; H_s]`, then 2 Mamba blocks.

> **F1 and D are statistically indistinguishable** (mission macro-F1 0.894 vs 0.893, with almost identical CIs).
> F1 is reported as the headline architecture because its **fusion mechanism is interpretable and content-adaptive**
> and ties directly to the project's audio/IMU complementarity evidence, **not because it measurably outperforms D.**
> This statement is regenerated from the loaded numbers wherever the two are compared.

**What each output needs:**

| needs per-window `.npz` files | uses the saved CSV / table values when `.npz` files are missing |
|---|---|
| confusion matrices, per-class precision and F1, raw safety counts, the g_a distribution, the complementarity figure | headline accuracy / macro-F1 / CI / balanced accuracy / AUROC / recall, safety shares, per-day scores, per-run gate statistics |

A figure that needs `.npz` files is **skipped with the missing file names printed**. It is never rebuilt from
aggregate numbers.

## 0. Setup — paths, the artifact search, v29 constants, helpers
Artifacts are searched for in this order:
1. `/kaggle/working`, if v29/v30 ran in the same session;
2. any attached `/kaggle/input/**` folder, e.g. v29's or v30's Kaggle notebook output added as an input;
3. the public project repo, cloned with `--depth 1`: `version 29/Results`, `version 29/Npz`, `version 30/...`.

In [ ]:
# ==============================================================================
# 0a. PATHS, PROJECT REPO, STYLE, HOOKS  (no torch import anywhere in this notebook)
# ==============================================================================
import ast, csv, gc, glob, hashlib, json, math, os, re, subprocess, sys, time, warnings
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

IS_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if IS_KAGGLE else Path.cwd() / "v31_outputs"
BASE = WORK / "tii"
FIG_DIR = WORK / "figures"
for _d in (WORK, BASE, FIG_DIR):
    _d.mkdir(parents=True, exist_ok=True)

PROJECT_REPO_URL = "https://github.com/Sainava/AeroSync-Mamba.git"
PROJECT_REPO = BASE / "AeroSync-Mamba"
if not (PROJECT_REPO / ".git").exists():
    _r = subprocess.run(["git", "clone", "--depth", "1", PROJECT_REPO_URL, str(PROJECT_REPO)], capture_output=True, text=True)
    print("project repo clone:", "OK" if _r.returncode == 0 else f"FAILED (non-fatal): {(_r.stderr or _r.stdout)[-200:]}")

SEARCH = {
    "v29_runs":  ["/kaggle/working/gated_fusion_results.csv", "/kaggle/input/**/gated_fusion_results.csv",
                  "{repo}/version 29/Results/gated_fusion_results.csv"],
    "v29_table": ["/kaggle/working/v29_results_table.csv", "/kaggle/input/**/v29_results_table.csv",
                  "{repo}/version 29/Results/v29_results_table.csv"],
    "v30_runs":  ["/kaggle/working/crossattn_results.csv", "/kaggle/input/**/crossattn_results.csv",
                  "{repo}/version 30/Results/crossattn_results.csv", "{repo}/version 30/**/crossattn_results.csv"],
    "v30_table": ["/kaggle/working/v30_results_table.csv", "/kaggle/input/**/v30_results_table.csv",
                  "{repo}/version 30/Results/v30_results_table.csv", "{repo}/version 30/**/v30_results_table.csv"],
    "v29_npz_dirs": ["/kaggle/working/v29_probs", "/kaggle/input/**/v29_probs", "{repo}/version 29/Npz", "{repo}/version 29/v29_probs"],
    "v30_npz_dirs": ["/kaggle/working/v30_probs", "/kaggle/input/**/v30_probs", "{repo}/version 30/Npz", "{repo}/version 30/v30_probs"],
}


def expand(pattern):
    p = pattern.replace("{repo}", str(PROJECT_REPO))
    return sorted(glob.glob(p, recursive=True)) if "*" in p else ([p] if Path(p).exists() else [])


def find_first(key):
    for pat in SEARCH[key]:
        hits = expand(pat)
        if hits:
            return Path(hits[0])
    return None


HOOKS = defaultdict(list)


def hook(name, ok, msg, stop=True):
    ok = bool(ok)
    HOOKS[name].append((ok, msg))
    print(f"[HOOK {name}] {'PASS' if ok else 'FAIL'} — {msg}")
    if not ok and stop:
        raise RuntimeError(f"HOOK {name} FAILED: {msg}")
    return ok


def loud(msg):
    print("🚨" * 50)
    for line in msg.split("\n"):
        print("🚨  " + line)
    print("🚨" * 50)


def sha16(*arrays):
    h = hashlib.sha1()
    for a in arrays:
        h.update(np.ascontiguousarray(a).tobytes())
    return h.hexdigest()[:16]


# ---- project figure style (v1-v5 report figures): Blues matrices, 300 dpi, PNG + PDF ----
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
CLASS_NAMES = ["0 (Normal)", "1 Broken", "2 Broken", "3 Broken", "4 Broken"]
PRF_COLORS = {"Precision": "#4c72b0", "Recall": "#55a868", "F1-Score": "#c44e52"}
MODEL_COLORS = {"F1": "#4c72b0", "D": "#8c8c8c", "A": "#dd8452", "I": "#55a868", "oracle": "#c44e52"}
PROTOCOL_TAG = "Day-out (leave-one-recording-day-out), 5 folds, 95 missions, seeds 42/123 (2) averaged per window"
FIGURES = []


def save_fig(fig, name, caption):
    """Every figure carries its protocol caption (under the axes) and is saved as PNG (300 dpi) + PDF."""
    assert "fold" in caption and ("seed" in caption or "Seed" in caption), f"{name}: caption must state folds and seeds"
    fig.text(0.5, -0.02, caption, ha="center", va="top", fontsize=8, color="#444444", wrap=True)
    for ext in ("png", "pdf"):
        fig.savefig(FIG_DIR / f"{name}.{ext}", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    FIGURES.append((name, caption))
    print(f"[FIGURE] {FIG_DIR / (name + '.png')} (+ .pdf) — {caption}")


def skip_fig(name, missing):
    print(f"[FIGURE SKIPPED] {name}: requires per-window probabilities; missing file(s): {missing}")
    FIGURES.append((name, "SKIPPED — missing " + ", ".join(missing[:6]) + (" ..." if len(missing) > 6 else "")))

In [ ]:
# ==============================================================================
# 0b. CONSTANTS — v29's protocol values (read by v29's verbatim data cells in section 1b) + reporting constants
# ==============================================================================
SEEDS = [42, 123]
V29_MODELS = ["A", "I", "D", "F0", "F1", "F2"]
V30_MODELS = ["CA", "CAG"]
ALL_MODELS = V29_MODELS + V30_MODELS
REQUIRED_NPZ_MODELS = ["F1", "D", "A", "I"]
N_CLASSES = 5
EXCLUDE_RATIO = 0.5
EXPECTED_EXCLUDED = {"rosbag2_2021_12_29-09_10_31", "rosbag2_2021_12_31-10_12_31",
                     "rosbag2_2021_12_31-09_58_13", "rosbag2_2021_12_31-10_04_22"}
EXPECTED_N_MISSIONS = 95
EXPECTED_CLASS_COUNTS = [18, 19, 18, 20, 20]
DAYS = ["2021-12-22", "2021-12-31", "2022-01-04", "2022-01-05", "2022-01-10"]
DAY_MERGE = {"2021-12-29": "2021-12-31"}
EXPECTED_DAY_COUNTS = [10, 36, 17, 15, 17]
SMALL_FOLD_DAY = "2021-12-22"
STEADY = (0.10, 0.90)
N_SLOTS, DROP_SLOT, FRAME = 6, 5, 256
KEEP_SLOTS = [s for s in range(N_SLOTS) if s != DROP_SLOT]
BUFFER_LEN = N_SLOTS * FRAME
KEEP_BINS = list(range(1, 17))
N_AUDIO_FEAT = len(KEEP_SLOTS) * len(KEEP_BINS)
WINDOW_SEC, STEP_SEC = 2.56, 1.28
TA_AUDIO, TS_SENSOR = 64, 128
MIN_SENSOR_SAMPLES_PER_WINDOW = 10
N_SENSOR_CHANNELS = 6
N_HASH_SAMPLE = 25
BOOTSTRAP_RESAMPLES = 2000
BOOTSTRAP_SEED = 2028
NOISE_FLOOR_F1 = 0.016

# --- Section 1 reproduction targets (known, verified v29 values; compared at 3 decimals) ---
KNOWN = {("F1", "window"): (0.869, 0.821, 0.913), ("F1", "mission"): (0.894, 0.832, 0.948),
         ("D", "window"): (0.871, 0.823, 0.917), ("D", "mission"): (0.893, 0.832, 0.948)}
V29_PAIRED_SHA = "44c04ca18ec7a131"
V29_N_WINDOWS = 5238
VERIFY_WINDOW_HASH = True     # rebuilds the paired windows with v29's verbatim cells (~5-8 min parse) to check the hash
CACHE_DIR = WORK / "v29_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"{PROTOCOL_TAG} | figures -> {FIG_DIR}")

In [ ]:
# ==============================================================================
# v29 helper functions, VERBATIM from version 29/minor-project-v29.ipynb (cell 18): compute_metrics, mission_level
# ==============================================================================
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score, roc_auc_score)
from scipy import stats
LABELS = list(range(N_CLASSES))


def compute_metrics(y, P):
    pred = P.argmax(1)
    try:
        auroc = float(roc_auc_score(y, P, multi_class="ovr", average="macro", labels=LABELS))
    except ValueError:
        auroc = float("nan")
    return {"accuracy": float(accuracy_score(y, pred)),
            "macro_f1": float(f1_score(y, pred, average="macro", labels=LABELS, zero_division=0)),
            "balanced_accuracy": float(balanced_accuracy_score(y, pred)), "macro_auroc": auroc}


def mission_level(missions, y, P):
    df = pd.DataFrame(P, columns=LABELS)
    df["mission"], df["y"] = missions, y
    g = df.groupby("mission", sort=True)
    return g["y"].first().values, g[LABELS].mean().values, g["y"].first().index.values

In [ ]:
# ==============================================================================
# v29 helper functions, VERBATIM from version 29/minor-project-v29.ipynb (cell 24): prf_from_cm, cluster_bootstrap_f1, level_block
# ==============================================================================



def prf_from_cm(cm):
    tp = np.diag(cm).astype(float)
    prec = np.divide(tp, cm.sum(0), out=np.zeros_like(tp), where=cm.sum(0) > 0)
    rec = np.divide(tp, cm.sum(1), out=np.zeros_like(tp), where=cm.sum(1) > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros_like(tp), where=(prec + rec) > 0)
    return prec, rec, f1


def cluster_bootstrap_f1(missions, y, pred, B=BOOTSTRAP_RESAMPLES, seed=BOOTSTRAP_SEED):
    """Macro-F1 over B resamples of MISSIONS within class (never windows)."""
    rng = np.random.default_rng(seed)
    ms = pd.Series(missions).unique()
    pos = {m: i for i, m in enumerate(ms)}
    cnt, mcls = np.zeros((len(ms), N_CLASSES), int), np.zeros(len(ms), int)
    for m, t, p in zip(missions, y, pred):
        cnt[pos[m], p] += 1
        mcls[pos[m]] = t
    cms = np.zeros((B, N_CLASSES, N_CLASSES))
    for c in range(N_CLASSES):
        rows = np.where(mcls == c)[0]
        cms[:, c, :] = cnt[rows[rng.integers(0, len(rows), (B, len(rows)))]].sum(1)
    return np.array([prf_from_cm(cm)[2].mean() for cm in cms])


def level_block(missions, y, P):
    pred = P.argmax(1)
    m = compute_metrics(y, P)
    cm = confusion_matrix(y, pred, labels=LABELS)
    _, rec, _ = prf_from_cm(cm)
    lo, hi = np.percentile(cluster_bootstrap_f1(missions, y, pred), [2.5, 97.5])
    err = pred != y
    return {"accuracy": m["accuracy"], "macro_f1": m["macro_f1"], "f1_ci_low": lo, "f1_ci_high": hi,
            "balanced_accuracy": m["balanced_accuracy"], "macro_auroc": m["macro_auroc"],
            **{f"recall_c{c}": rec[c] for c in range(N_CLASSES)},
            "faulty_pred_normal": float((pred[y > 0] == 0).mean()), "normal_pred_faulty": float((pred[y == 0] > 0).mean()),
            "adjacent_error_share": float((np.abs(pred - y)[err] == 1).mean()) if err.any() else float("nan"), "cm": cm}

## 1. Setup and reproduction check
- **1a:** load v29's per-run CSV and pooled table, v30's if present, and every per-window `.npz` found. Print exactly
  which expected files are missing.
- **Reproduction check:**
  - F1's and D's pooled day-out macro-F1 and 95 % CIs, at window and mission level, must match the known values at 3
    decimals: F1 0.869 [0.821, 0.913] / 0.894 [0.832, 0.948] and D 0.871 [0.823, 0.917] / 0.893 [0.832, 0.948].
  - If the `.npz` files are present, the same numbers are also recomputed from them and must equal the table
    **exactly**.
  - **A FAIL stops the notebook here**, before any figure is drawn.
- **1b:** the 95-mission paired-window set is rebuilt with **v29's verbatim cells** and its hash is checked against
  `44c04ca18ec7a131`. The `.npz` window ids must be exactly that set.

In [ ]:
# ==============================================================================
# 1a. LOAD SAVED RESULTS + REPRODUCTION CHECK (stops on FAIL)
# ==============================================================================
P_V29_RUNS, P_V29_TABLE = find_first("v29_runs"), find_first("v29_table")
P_V30_RUNS, P_V30_TABLE = find_first("v30_runs"), find_first("v30_table")
if P_V29_RUNS is None or P_V29_TABLE is None:
    loud("v29's gated_fusion_results.csv / v29_results_table.csv not found. Searched:\n"
         + "\n".join(p.replace("{repo}", str(PROJECT_REPO)) for k in ("v29_runs", "v29_table") for p in SEARCH[k]))
    raise RuntimeError("v29 results are required.")
V29_RUNS, V29_TABLE = pd.read_csv(P_V29_RUNS), pd.read_csv(P_V29_TABLE)
V30_RUNS = pd.read_csv(P_V30_RUNS) if P_V30_RUNS else None
V30_TABLE = pd.read_csv(P_V30_TABLE) if P_V30_TABLE else None
print(f"v29 runs:  {P_V29_RUNS} ({len(V29_RUNS)} rows)\nv29 table: {P_V29_TABLE}\n"
      f"v30 runs:  {P_V30_RUNS or 'NOT FOUND'}\nv30 table: {P_V30_TABLE or 'NOT FOUND (appendix will show the 6 v29 models only)'}")
hook("inputs", {(r.model, int(r.fold), int(r.seed)) for r in V29_RUNS.itertuples()} ==
     {(m, k, s) for m in V29_MODELS for k in range(len(DAYS)) for s in SEEDS},
     "v29 per-run CSV holds exactly 6 models x 5 folds x seeds 42/123 (60 runs)")

# ---- per-window probability files ----
_NPZ_RE = re.compile(r"^(A|I|D|F0|F1|F2|CA|CAG)_(\d)_(\d+)\.npz$")
NPZ, _searched = {}, []
for key in ("v29_npz_dirs", "v30_npz_dirs"):
    for pat in SEARCH[key]:
        _searched.append(pat.replace("{repo}", str(PROJECT_REPO)))
        for d in expand(pat):
            for f in sorted(Path(d).glob("*.npz")):
                m_ = _NPZ_RE.match(f.name)                      # ignores copies such as "I_2_123 (1).npz"
                if m_:
                    NPZ.setdefault((m_.group(1), int(m_.group(2)), int(m_.group(3))), f)
MISSING = {m: [f"{m}_{k}_{s}.npz" for k in range(len(DAYS)) for s in SEEDS if (m, k, s) not in NPZ] for m in ALL_MODELS}
HAVE = {m: not MISSING[m] for m in ALL_MODELS}
print("\nPer-window probability files (expected 10 per model: {model}_{fold}_{seed}.npz, folds 0-4, seeds 42/123):")
for m in ALL_MODELS:
    print(f"   {m:<4} {10 - len(MISSING[m]):>2}/10" + ("" if HAVE[m] else f"   MISSING: {MISSING[m]}"))
if not all(HAVE[m] for m in REQUIRED_NPZ_MODELS):
    loud("Per-window .npz files needed for confusion matrices, per-class precision/F1, raw safety counts, the g_a "
         "distribution and the complementarity figure are missing for: "
         + ", ".join(m for m in REQUIRED_NPZ_MODELS if not HAVE[m]) + "\nSearched (in order):\n   " + "\n   ".join(_searched)
         + "\nThose figures will be SKIPPED with the file names printed. To enable them: attach v29's Kaggle output (v29_probs/) "
           "or push its .npz files to 'version 29/Npz/' in the project repo.")


def load_npz(m, k, s):
    z = np.load(NPZ[(m, k, s)])
    return {key: z[key] for key in z.files}


def pooled(m):
    """Seed-averaged per-window probabilities over the 5 test folds; per-window metadata from the files themselves."""
    per_seed = []
    for s in SEEDS:
        parts = [load_npz(m, k, s) for k in range(len(DAYS))]
        per_window = [key for key in parts[0] if all(p[key].shape[:1] == p["window_id"].shape for p in parts)]   # not audio_mask
        cat = {key: np.concatenate([p[key] for p in parts]) for key in per_window}
        o = np.argsort(cat["window_id"])
        per_seed.append({key: v[o] for key, v in cat.items()})
    for ps in per_seed[1:]:
        assert np.array_equal(ps["window_id"], per_seed[0]["window_id"]) and np.array_equal(ps["cls"], per_seed[0]["cls"])
    meta = pd.DataFrame({key: per_seed[0][key] for key in ("window_id", "mission", "day", "cls", "phase")})
    out = {"meta": meta, "P": np.mean([ps["probs"] for ps in per_seed], axis=0), "per_seed": per_seed}
    my, mP, mnames = mission_level(meta.mission.values, meta.cls.values, out["P"])
    out.update({"m_y": my, "m_P": mP, "m_names": mnames})
    return out


POOL = {m: pooled(m) for m in ALL_MODELS if HAVE[m]}


def table_row(m, level):
    src = V29_TABLE if m in V29_MODELS else V30_TABLE
    if src is None:
        return None
    r = src[(src.model == m) & (src.level == level)]
    return None if r.empty else r.iloc[0]


# ---- reproduction check ----
_ok_all = True
for (m, level), (f, lo, hi) in KNOWN.items():
    r = table_row(m, level)
    got = (round(float(r.macro_f1), 3), round(float(r.f1_ci_low), 3), round(float(r.f1_ci_high), 3))
    ok = got == (f, lo, hi)
    _ok_all &= ok
    hook("reproduction", ok, f"{m} {level}: loaded {got[0]:.3f} [{got[1]:.3f}, {got[2]:.3f}] vs known {f:.3f} [{lo:.3f}, {hi:.3f}]",
         stop=False)
    if m in POOL:
        p = POOL[m]
        b = level_block(p["meta"].mission.values, p["meta"].cls.values, p["P"]) if level == "window" else \
            level_block(p["m_names"], p["m_y"], p["m_P"])
        ok2 = all(abs(b[k_] - float(r[k_])) < 1e-9 for k_ in ("macro_f1", "f1_ci_low", "f1_ci_high", "accuracy", "balanced_accuracy"))
        _ok_all &= ok2
        hook("reproduction", ok2, f"{m} {level}: recomputed from the .npz files {b['macro_f1']:.6f} [{b['f1_ci_low']:.6f}, "
                                  f"{b['f1_ci_high']:.6f}] == table exactly", stop=False)
    else:
        print(f"   ({m} {level}: .npz files missing — recomputation not possible; table values checked only)")
if not _ok_all:
    loud("REPRODUCTION CHECK FAILED — the loaded results do not match F1's / D's known day-out values.\n"
         "No figure is produced from stale or mismatched data. Check which files were loaded above.")
    raise RuntimeError("Section 1 reproduction check FAILED.")
print("\nSection 1 reproduction check: PASS")


def tie_note():
    f, d = table_row("F1", "mission"), table_row("D", "mission")
    return (f"F1 and D are statistically indistinguishable (mission macro-F1 {f.macro_f1:.3f} [{f.f1_ci_low:.3f}, "
            f"{f.f1_ci_high:.3f}] vs {d.macro_f1:.3f} [{d.f1_ci_low:.3f}, {d.f1_ci_high:.3f}]). F1 is reported as the headline "
            f"architecture because it has an interpretable, content-adaptive fusion mechanism tied to the project's "
            f"complementarity evidence — not because it measurably outperforms D.")


print("\nNOTE: " + tie_note())

In [ ]:
# ==============================================================================
# 1b. PAIRED-WINDOW SET == v29's (hash 44c04ca18ec7a131): rebuilt with v29's VERBATIM data cells below
# ==============================================================================
if VERIFY_WINDOW_HASH:
    def _is_dataset_root(p):
        return p.is_dir() and all((p / str(c)).is_dir() for c in range(5))

    _att = []
    for _pat in ("/kaggle/input/*/Dataset", "/kaggle/input/*/*/Dataset", "/kaggle/input/*/*/*/Dataset"):
        _att += [Path(p) for p in sorted(glob.glob(_pat)) if _is_dataset_root(Path(p))]
    if _att:
        DATA_ROOT = _att[0]
    else:
        _repo = BASE / "UAV-Realistic-Fault-Dataset"
        if not _is_dataset_root(_repo / "Dataset"):
            print("Cloning the TII dataset (--depth 1) for the window-hash check ...")
            subprocess.run(["git", "clone", "--depth", "1", "https://github.com/tiiuae/UAV-Realistic-Fault-Dataset.git", str(_repo)],
                           check=True)
        DATA_ROOT = _repo / "Dataset"
    print(f"DATA_ROOT {DATA_ROOT} — running v29's parse / metadata / paired-window cells verbatim (next three cells)")
else:
    print("VERIFY_WINDOW_HASH = False: the three v29 data cells below are skipped and the window-hash hook is NOT run.")

In [ ]:
# v29 cell 8, VERBATIM (indented under the VERIFY_WINDOW_HASH flag only)
if VERIFY_WINDOW_HASH:
    # ==============================================================================
    # 3a. PARSE (all 99 folders: the 4 exclusions are computed from metadata, not assumed) -> v29_cache/<mission>.npz
    # ==============================================================================
    def unwrap_message(obj):                                   # IMU notebook, unchanged
        if isinstance(obj, dict):
            for key in ("msg", "message", "data"):
                if key in obj and isinstance(obj[key], dict):
                    return obj[key]
        return obj


    def get_vec(d, candidates):                                # IMU notebook, unchanged
        for key in candidates:
            if key in d:
                v = d[key]
                if isinstance(v, (list, tuple)) and len(v) >= 3:
                    return np.asarray(v[:3], dtype=np.float64)
            vals, ok = [], True
            for i in range(3):
                k = key.format(i=i)
                if k in d:
                    vals.append(d[k])
                else:
                    ok = False
                    break
            if ok:
                return np.asarray(vals, dtype=np.float64)
        return None


    def parse_imu_file(path):
        """IMU notebook's parser: rows [timestamp_us, acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z] in FILE ORDER."""
        rows, bad = [], 0
        with open(path, "r") as f:
            for line in f:
                try:
                    obj = unwrap_message(json.loads(line))
                    ts = obj.get("timestamp", obj.get("time_stamp", None))
                    acc = get_vec(obj, ["accelerometer_m_s2", "accelerometer_m_s2[{i}]", "accelerometer", "acceleration"])
                    gyro = get_vec(obj, ["gyro_rad", "gyro_rad[{i}]", "gyro", "angular_velocity"])
                    if ts is None or acc is None or gyro is None:
                        bad += 1
                        continue
                    rows.append([float(ts), *acc.tolist(), *gyro.tolist()])
                except Exception:
                    bad += 1
        return np.asarray(rows, dtype=np.float64).reshape(-1, 7), bad


    def parse_mission(mdir):
        cache = CACHE_DIR / f"{mdir.name}.npz"
        if cache.exists():
            z = np.load(cache)
            return {k: z[k] for k in z.files}
        s_path = sorted(mdir.glob("*SensorCombined*.jsonl"))[0]
        a_path = sorted(mdir.glob("*AudioBuffer*.jsonl"))[0]
        imu, bad = parse_imu_file(s_path)
        chunks, lens, rst, rsts = [], Counter(), None, set()
        with open(a_path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    continue
                x = rec.get("data")
                if x is None or len(x) == 0:
                    continue
                if "recording_started_time" in rec:
                    rsts.add(float(rec["recording_started_time"]))
                    if rst is None:
                        rst = float(rec["recording_started_time"])          # v24: the FIRST record's value
                lens[len(x)] += 1
                chunks.append(x)
        modal = lens.most_common(1)[0][0]
        assert modal == BUFFER_LEN, f"{mdir.name}: modal buffer length {modal}"
        assert rst is not None, f"{mdir.name}: no recording_started_time — the audio cannot be anchored"
        keep_idx = np.array([i for i, x in enumerate(chunks) if len(x) == modal], dtype=np.int64)
        raw = np.asarray([chunks[i] for i in keep_idx], dtype=np.int64)
        n_all = len(chunks)
        del chunks
        frames = raw.reshape(len(raw), FRAME, N_SLOTS)                # sample i -> frame i // 6, slot i % 6
        slot6_zero = bool(np.all(frames[:, :, DROP_SLOT] == 0))
        ch = frames[:, :, KEEP_SLOTS].astype(np.float64)
        ch = ch - ch.mean(axis=1, keepdims=True)                      # per-chunk mean removed, per channel
        pw = (np.abs(np.fft.rfft(ch, axis=1)) ** 2).transpose(0, 2, 1).astype(np.float32)    # (n, 5, 129)
        feats = (pw[:, :, KEEP_BINS] ** 0.25).reshape(len(pw), N_AUDIO_FEAT).astype(np.float32)   # sqrt|rFFT|, channel-major
        out = {"imu_t": imu[:, 0], "imu_x": imu[:, 1:].astype(np.float32), "imu_bad": bad, "feats": feats,
               "keep_idx": keep_idx, "n_all_chunks": n_all, "rst": rst, "n_rst_values": len(rsts), "slot6_zero": slot6_zero}
        tmp = CACHE_DIR / f"{mdir.name}.tmp.npz"
        np.savez(tmp, **out)
        os.replace(tmp, cache)
        return out


    def mission_day(name):
        d = name[8:18].replace("_", "-")
        return DAY_MERGE.get(d, d)


    ALL_DIRS, LABEL_OF = [], {}
    for class_dir in sorted(DATA_ROOT.iterdir()):
        if not (class_dir.is_dir() and class_dir.name.isdigit()):
            continue
        for mdir in sorted(class_dir.iterdir()):
            if mdir.is_dir() and not mdir.name.startswith("."):
                ALL_DIRS.append(mdir)
                LABEL_OF[mdir.name] = int(class_dir.name)
    _t0 = time.time()
    PARSED = {}
    for i, mdir in enumerate(ALL_DIRS, 1):
        PARSED[mdir.name] = parse_mission(mdir)
        if i % 10 == 0 or i == len(ALL_DIRS):
            print(f"  parsed {i}/{len(ALL_DIRS)} missions | {time.time() - _t0:.0f} s")

In [ ]:
# v29 cell 9, VERBATIM (indented under the VERIFY_WINDOW_HASH flag only)
if VERIFY_WINDOW_HASH:
    # ==============================================================================
    # 3b. METADATA — v24's corrected audio anchor, exclusion rule, mission / class / day counts (HOOK data)
    # ==============================================================================
    def timebase(P):
        """v24 CORRECTED_ALIGNMENT time axes. IMU: FILE ORDER, seconds from the first sample. Audio: anchored at
        recording_started_time; chunk duration = (mission duration - offset) / number of chunks."""
        t_us = P["imu_t"]
        s_times = (t_us - t_us[0]) / 1e6
        duration = float(s_times[-1] - s_times[0])
        offset = (float(P["rst"]) - float(t_us[0])) / 1e6
        chunk = (duration - offset) / int(P["n_all_chunks"])
        a_times = (offset + np.arange(int(P["n_all_chunks"]), dtype=np.float64) * chunk)[P["keep_idx"]]
        return s_times, a_times, duration, offset, chunk


    META = []
    for name, P in PARSED.items():
        s_times, a_times, dur, off, ch = timebase(P)
        META.append({"mission": name, "class": LABEL_OF[name], "day": mission_day(name), "duration_s": dur, "offset_s": off,
                     "ratio": off / dur, "chunk_ms": 1e3 * ch, "n_imu": len(s_times), "imu_inversions": int((np.diff(s_times) < 0).sum()),
                     "n_chunks": int(P["n_all_chunks"]), "chunks_dropped": int(P["n_all_chunks"]) - len(P["keep_idx"]),
                     "rst_values": int(P["n_rst_values"]), "slot6_zero": bool(P["slot6_zero"])})
    META = pd.DataFrame(META).set_index("mission")
    EXCLUDED = set(META.index[META.ratio >= EXCLUDE_RATIO])
    KEPT = sorted(set(META.index) - EXCLUDED)
    M = META.loc[KEPT]
    LABEL = {n: int(M.loc[n, "class"]) for n in KEPT}
    DAY = {n: M.loc[n, "day"] for n in KEPT}
    _cls = np.bincount(M["class"], minlength=N_CLASSES).tolist()
    _days = [int((M.day == d).sum()) for d in DAYS]
    print(META.loc[sorted(EXCLUDED), ["class", "day", "offset_s", "duration_s", "ratio"]].round(3).to_string())
    print(f"\nKept {len(KEPT)} missions | max kept ratio {M.ratio.max():.3f} | median offset {M.offset_s.median():.2f} s | "
          f"chunk duration {M.chunk_ms.median():.2f} ms (median; v24's corrected-anchor formula) | "
          f"non-modal chunks dropped {int(M.chunks_dropped.sum())} | missions with IMU timestamp inversions "
          f"{int((M.imu_inversions > 0).sum())} (total {int(M.imu_inversions.sum())})")
    print(pd.crosstab(M.day, M["class"]).reindex(DAYS).to_string())
    hook("data", EXCLUDED == EXPECTED_EXCLUDED, f"exclusion set == the 4 known names: {sorted(EXCLUDED)}")
    hook("data", len(KEPT) == EXPECTED_N_MISSIONS and _cls == EXPECTED_CLASS_COUNTS and _days == EXPECTED_DAY_COUNTS,
         f"{len(KEPT)} missions, classes {_cls}, days {_days} (expected {EXPECTED_N_MISSIONS}, {EXPECTED_CLASS_COUNTS}, {EXPECTED_DAY_COUNTS})")
    hook("data", bool(M.slot6_zero.all()) and bool((M.rst_values == 1).all()),
         f"audio slot 6 identically zero in {int(M.slot6_zero.sum())}/{len(M)}; recording_started_time constant in "
         f"{int((M.rst_values == 1).sum())}/{len(M)}")

In [ ]:
# v29 cell 10, VERBATIM (indented under the VERIFY_WINDOW_HASH flag only)
if VERIFY_WINDOW_HASH:
    # ==============================================================================
    # 3c. ONE SET OF PAIRED WINDOWS (v24 CORRECTED_ALIGNMENT): audio (64 x 80) and IMU (128 x 6, file order) from the SAME
    #     real time span of the SAME mission. Every model below reads these arrays (unimodal models ignore one half).
    # ==============================================================================
    def paired_windows(name, P):
        s_times, a_times, dur, _, _ = timebase(P)
        tel, feats = P["imu_x"], P["feats"]
        max_time = min(s_times[-1], a_times[-1])
        out, i = [], 0
        t_start = max(0.0, float(a_times[0]))                  # v24: CORRECTED_WINDOW_START = "audio_onset"
        while t_start + WINDOW_SEC <= max_time:
            t_end = t_start + WINDOW_SEC
            s_mask = (s_times >= t_start) & (s_times <= t_end)
            if s_mask.sum() < MIN_SENSOR_SAMPLES_PER_WINDOW:
                t_start += STEP_SEC
                continue
            s_idx = np.where(s_mask)[0]
            pick = s_idx[np.linspace(0, len(s_idx) - 1, TS_SENSOR).round().astype(int)]
            tau_s_abs = s_times[pick]
            x_s = tel[pick].astype(np.float32)                 # real file-order samples (IMU notebook rule)
            x_s_v24 = np.stack([np.interp(tau_s_abs, s_times[s_mask], tel[s_mask, c]) for c in range(N_SENSOR_CHANNELS)],
                               axis=1).astype(np.float32)      # v24's np.interp form, kept only for the comparison count
            a_mask = (a_times >= t_start) & (a_times <= t_end)
            if a_mask.sum() >= 2:
                a_idx = np.where(a_mask)[0]
                a_pick = a_idx[np.linspace(0, len(a_idx) - 1, TA_AUDIO).round().astype(int)]
                tau_a_abs = a_times[a_pick]
                x_a = feats[a_pick].astype(np.float32)         # == v24's np.interp at exact (strictly increasing) chunk times
            else:
                tau_a_abs = np.linspace(t_start, t_end, TA_AUDIO)
                x_a = np.stack([np.interp(tau_a_abs, a_times, feats[:, c], left=0.0, right=0.0) for c in range(N_AUDIO_FEAT)],
                               axis=1).astype(np.float32)
            rel = (t_start + WINDOW_SEC / 2) / dur
            out.append({"window_id": f"{name}#{i:04d}", "mission": name, "cls": LABEL[name], "day": DAY[name],
                        "t_start": t_start, "t_end": t_end,
                        "phase": "start" if rel < STEADY[0] else ("end" if rel > STEADY[1] else "steady"),
                        "x_a": x_a, "x_s": x_s, "tau_a": ((tau_a_abs - t_start) / WINDOW_SEC).astype(np.float32),
                        "tau_s": ((tau_s_abs - t_start) / WINDOW_SEC).astype(np.float32),
                        "v24_interp_differs": bool(not np.array_equal(x_s, x_s_v24)),
                        "v24_interp_maxdiff": float(np.abs(x_s - x_s_v24).max())})
            i += 1
            t_start += STEP_SEC                                 # v24: accumulated float step
        return out


    _W = []
    for name in KEPT:
        _W.extend(paired_windows(name, PARSED[name]))
    WMETA = pd.DataFrame([{k: v for k, v in w.items() if k not in ("x_a", "x_s", "tau_a", "tau_s")} for w in _W])
    XA = np.stack([w["x_a"] for w in _W])                      # (N, 64, 80)
    XS = np.stack([w["x_s"] for w in _W])                      # (N, 128, 6)
    TAU_A = np.stack([w["tau_a"] for w in _W])                 # (N, 64)   in window units (v24 dataset: tau / window_sec)
    TAU_S = np.stack([w["tau_s"] for w in _W])                 # (N, 128)
    YW = WMETA.cls.values.astype(np.int64)
    del _W
    gc.collect()
    print(f"Paired windows: {len(WMETA)} | XA {XA.shape} | XS {XS.shape} | per class {np.bincount(YW, minlength=N_CLASSES).tolist()} | "
          f"per phase {WMETA.phase.value_counts().to_dict()}")
    print(f"IMU: windows where the file-order direct pick differs from v24's np.interp form: "
          f"{int(WMETA.v24_interp_differs.sum())} / {len(WMETA)} (max abs diff {WMETA.v24_interp_maxdiff.max():.4g}) — "
          "nonzero only where a window's IMU timestamps go backwards")

    # ---- pairing hooks ----
    _span_ok = bool((TAU_A >= -1e-6).all() and (TAU_A <= 1 + 1e-6).all() and (TAU_S >= -1e-6).all() and (TAU_S <= 1 + 1e-6).all())
    hook("pairing", _span_ok and WMETA.window_id.is_unique,
         "every window's audio tokens AND IMU samples lie inside its own [t_start, t_start + 2.56 s] span of ONE mission "
         "(both come from a single record: same mission id, same span)")
    _rng_h = np.random.default_rng(0)
    _sample = np.sort(_rng_h.choice(len(WMETA), size=min(N_HASH_SAMPLE, len(WMETA)), replace=False))
    _hash_ok = True
    for j in _sample:
        r = WMETA.iloc[j]
        rebuilt = {w["window_id"]: w for w in paired_windows(r.mission, PARSED[r.mission])}[r.window_id]
        same = sha16(rebuilt["x_a"], rebuilt["x_s"]) == sha16(XA[j], XS[j]) and abs(rebuilt["t_start"] - r.t_start) < 1e-12
        _hash_ok &= same
    hook("pairing", _hash_ok, f"hash check on {len(_sample)} sampled windows: stored (audio, IMU) == rebuilt from the raw "
                              f"mission at the recorded (mission, t_start)")
    _fin = bool(np.isfinite(XA).all() and np.isfinite(XS).all())
    hook("pairing", _fin and XA.shape[1:] == (TA_AUDIO, N_AUDIO_FEAT) and XS.shape[1:] == (TS_SENSOR, N_SENSOR_CHANNELS),
         f"shapes audio {XA.shape[1:]}, IMU {XS.shape[1:]}; all finite: {_fin}")
    PAIRED_SHA = sha16(XA, XS)
    print(f"Paired-window content sha1: {PAIRED_SHA}")

In [ ]:
# ==============================================================================
# 1c. WINDOW-SET HOOKS
# ==============================================================================
if VERIFY_WINDOW_HASH:
    hook("windows", PAIRED_SHA == V29_PAIRED_SHA and len(WMETA) == V29_N_WINDOWS,
         f"paired-window content sha1 {PAIRED_SHA} == v29's {V29_PAIRED_SHA}; {len(WMETA)} windows")
    hook("windows", len(KEPT) == EXPECTED_N_MISSIONS and _cls == EXPECTED_CLASS_COUNTS and _days == EXPECTED_DAY_COUNTS,
         f"{len(KEPT)} missions, classes {_cls}, days {_days}")
    _wm = WMETA.set_index("window_id")
    for m, p in POOL.items():
        ids = p["meta"].window_id.values
        ok = (np.array_equal(ids, np.sort(WMETA.window_id.values))
              and np.array_equal(_wm.loc[ids, "mission"].values, p["meta"].mission.values)
              and np.array_equal(_wm.loc[ids, "cls"].values, p["meta"].cls.values))
        hook("windows", ok, f"{m}: .npz window ids / missions / classes == the verified 5238-window, 95-mission set")
else:
    print("window-hash hook not run (VERIFY_WINDOW_HASH = False)")
_idsets = {m: tuple(p["meta"].window_id.values) for m, p in POOL.items()}
if _idsets:
    hook("windows", len(set(_idsets.values())) == 1, f"all loaded models share one window set ({list(_idsets)})")

## 2. Headline metrics — F1 (headline) and D (comparison)
The table gives accuracy, macro-F1 with its 95 % mission-level bootstrap CI, balanced accuracy and macro-AUROC, at
window and mission level. These come from v29's pooled table, the same values verified in section 1.

Per-class precision / recall / F1 are computed from the per-window files; without them, only per-class recall (from
the table) is shown. The table is printed as markdown and saved as a PNG and a CSV. F1 − D per fold is also given
under **both** paired-statistics methods (v30's fix).

In [ ]:
# ==============================================================================
# 2. HEADLINE TABLE (markdown + PNG + CSV) and F1 − D paired fold statistics (both methods)
# ==============================================================================
from scipy import stats


def md_table(df, floatfmt="{:.3f}"):
    cols = [df.index.name or ""] + [str(c) for c in df.columns]
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    for idx, row in df.iterrows():
        cells = [str(idx)] + [floatfmt.format(v) if isinstance(v, (float, np.floating)) else str(v) for v in row.values]
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)


def render_table_png(df, name, title, caption, col_width=1.45):
    fig, ax = plt.subplots(figsize=(max(6, col_width * (len(df.columns) + 1)), 0.45 * (len(df) + 2)))
    ax.axis("off")
    cells = [[f"{v:.3f}" if isinstance(v, (float, np.floating)) else str(v) for v in row] for row in df.values]
    t = ax.table(cellText=cells, rowLabels=[str(i) for i in df.index], colLabels=[str(c) for c in df.columns],
                 loc="center", cellLoc="center")
    t.auto_set_font_size(False)
    t.set_fontsize(8)
    t.scale(1, 1.4)
    for (r_, c_), cell in t.get_celld().items():
        if r_ == 0 or c_ == -1:
            cell.set_text_props(fontweight="bold")
            cell.set_facecolor("#e8eef7")
    ax.set_title(title, fontsize=11, fontweight="bold", pad=10)
    save_fig(fig, name, caption)


HEAD = []
for m in ("F1", "D"):
    for level in ("window", "mission"):
        r = table_row(m, level)
        HEAD.append({"model / level": f"{m} — {level}", "accuracy": float(r.accuracy),
                     "macro-F1 [95% CI]": f"{r.macro_f1:.3f} [{r.f1_ci_low:.3f}, {r.f1_ci_high:.3f}]",
                     "balanced acc.": float(r.balanced_accuracy), "macro-AUROC": float(r.macro_auroc)})
HEAD = pd.DataFrame(HEAD).set_index("model / level")
print("HEADLINE — pooled day-out (" + PROTOCOL_TAG + ")\n")
print(md_table(HEAD))
HEAD.to_csv(WORK / "v31_headline_table.csv")
render_table_png(HEAD, "t1_headline_F1_vs_D", "Headline day-out results — F1 (headline) and D (comparison)",
                 f"{PROTOCOL_TAG}. CI = mission-level cluster bootstrap (2000 resamples within class). {tie_note()}")

PERCLASS = {}
for m in ("F1", "D"):
    for level in ("window", "mission"):
        if m in POOL:
            p = POOL[m]
            y, pred = (p["meta"].cls.values, p["P"].argmax(1)) if level == "window" else (p["m_y"], p["m_P"].argmax(1))
            pr, rc, f1 = prf_from_cm(confusion_matrix(y, pred, labels=LABELS))
            PERCLASS[(m, level)] = pd.DataFrame({"precision": pr, "recall": rc, "F1": f1,
                                                 "support": np.bincount(y, minlength=N_CLASSES)}, index=CLASS_NAMES)
        else:
            r = table_row(m, level)
            PERCLASS[(m, level)] = pd.DataFrame({"precision": ["needs .npz"] * N_CLASSES,
                                                 "recall": [float(r[f"recall_c{c}"]) for c in range(N_CLASSES)],
                                                 "F1": ["needs .npz"] * N_CLASSES}, index=CLASS_NAMES)
            print(f"({m} {level}: per-class precision/F1 need {MISSING[m]}; recall shown from v29's table)")
for (m, level), t in PERCLASS.items():
    print(f"\nPer-class — {m}, {level} level\n" + md_table(t))
    t.to_csv(WORK / f"v31_perclass_{m}_{level}.csv")
if all(m in POOL for m in ("F1", "D")):
    both = pd.concat({f"F1 {lv}": PERCLASS[("F1", lv)][["precision", "recall", "F1"]] for lv in ("window", "mission")} |
                     {f"D {lv}": PERCLASS[("D", lv)][["precision", "recall", "F1"]] for lv in ("window", "mission")}, axis=1)
    both.columns = [f"{a}: {b}" for a, b in both.columns]
    render_table_png(both, "t2_perclass_F1_vs_D", "Per-class precision / recall / F1 — F1 vs D",
                     f"{PROTOCOL_TAG}. {tie_note()}", col_width=1.0)
else:
    skip_fig("t2_perclass_F1_vs_D", MISSING["F1"] + MISSING["D"])


# ---- F1 − D per fold, both paired-statistics methods ----
def fold_seedavg(m, col):
    runs = V29_RUNS if m in V29_MODELS else V30_RUNS
    d = runs[runs.model == m]
    return np.array([d[d.fold == k][col].mean() for k in range(len(DAYS))])


def fold_pooled(m, level, metric="macro_f1"):
    if m not in POOL:
        return None
    p, out = POOL[m], []
    for day in DAYS:
        sel = (p["meta"].day == day).values
        y, P = p["meta"].cls.values[sel], p["P"][sel]
        if level == "mission":
            y, P, _ = mission_level(p["meta"].mission.values[sel], y, P)
        out.append(compute_metrics(y, P)[metric])
    return np.array(out)


def paired_both(a, b, level="window"):
    col = "win_macro_f1" if level == "window" else "mis_macro_f1"
    res = {}
    for meth, (sa, sb) in (("pooled-probability", (fold_pooled(a, level), fold_pooled(b, level))),
                           ("seed-averaged-F1", (fold_seedavg(a, col), fold_seedavg(b, col)))):
        if sa is None or sb is None:
            res[meth] = None
            continue
        dlt = sa - sb
        t_, p_ = stats.ttest_rel(sa, sb)
        res[meth] = {"mean": dlt.mean(), "sd": dlt.std(ddof=1), "wins": int((dlt > 0).sum()), "p": p_, "diffs": np.round(dlt, 4)}
    pp, sf = res["pooled-probability"], res["seed-averaged-F1"]
    flag = ("ONE METHOD ONLY (pooled-probability needs .npz)" if pp is None or sf is None else
            "METHOD DISAGREEMENT" if pp["wins"] != sf["wins"] or np.sign(pp["mean"]) != np.sign(sf["mean"]) else "methods agree")
    return res, flag


for level in ("window", "mission"):
    res, flag = paired_both("F1", "D", level)
    print(f"\nF1 − D per fold [{level}] ({flag}):")
    for meth, s in res.items():
        print(f"   {meth:<20} " + ("UNAVAILABLE" if s is None else
                                   f"mean Δ {s['mean']:+.4f}  SD {s['sd']:.4f}  F1 wins {s['wins']}/5  paired t p {s['p']:.3f}  "
                                   f"per fold {s['diffs'].tolist()}"))
print("\n" + tie_note())

## 3. Confusion matrices — F1 (primary) and D (comparison)
The matrices are pooled over the 5 unseen-day folds from the seed-averaged probabilities. Rows = true class,
columns = predicted class; the counts are annotated in the project's `Blues` style. Four figures are produced, one
per (model, level). Each has two panels: raw counts and the row-normalized (recall) matrix. The adjacent-vs-distant
error breakdown is printed for both models.

In [ ]:
# ==============================================================================
# 3. CONFUSION MATRICES (needs .npz) + adjacent vs distant errors
# ==============================================================================
def cm_panel(ax, M, title, fmt):
    im = ax.imshow(M, cmap="Blues", interpolation="nearest", vmin=0)
    ax.grid(False)
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            ax.text(j, i, fmt(M[i, j]), ha="center", va="center", fontsize=9,
                    color="black" if M[i, j] < M.max() / 2 else "white")
    ax.set_xticks(range(N_CLASSES))
    ax.set_yticks(range(N_CLASSES))
    ax.set_xticklabels(range(N_CLASSES))
    ax.set_yticklabels(range(N_CLASSES))
    ax.set_xlabel("Predicted Class (Broken Propellers)", fontsize=10)
    ax.set_ylabel("True Class (Broken Propellers)", fontsize=10)
    ax.set_title(title, fontsize=10, fontweight="bold")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)


CMS = {}
for m in ("F1", "D"):
    for level in ("window", "mission"):
        name = f"f3_confusion_{m}_{level}"
        if m not in POOL:
            skip_fig(name, MISSING[m])
            continue
        p = POOL[m]
        y, pred = (p["meta"].cls.values, p["P"].argmax(1)) if level == "window" else (p["m_y"], p["m_P"].argmax(1))
        cm = confusion_matrix(y, pred, labels=LABELS)
        CMS[(m, level)] = cm
        fig, axs = plt.subplots(1, 2, figsize=(11, 4.6), dpi=150)
        cm_panel(axs[0], cm, f"{m} — {level} level: counts", lambda v: f"{int(v)}")
        cm_panel(axs[1], cm / np.maximum(cm.sum(1, keepdims=True), 1), f"{m} — {level} level: row-normalized (recall)",
                 lambda v: f"{v:.2f}")
        fig.suptitle(f"{'F1 (headline)' if m == 'F1' else 'D (comparison)'} — pooled day-out confusion matrix, {level} level",
                     fontsize=11, fontweight="bold")
        fig.tight_layout()
        save_fig(fig, name, f"{PROTOCOL_TAG}; {int(cm.sum())} {level}s. Rows = true, columns = predicted."
                            + (f" {tie_note()}" if m == "D" else ""))

print("\nAdjacent vs distant errors (share of misclassified windows / missions):")
for m in ("F1", "D"):
    for level in ("window", "mission"):
        if (m, level) in CMS:
            cm = CMS[(m, level)]
            off = [(i, j, cm[i, j]) for i in range(N_CLASSES) for j in range(N_CLASSES) if i != j]
            n_err = sum(v for _, _, v in off)
            adj = sum(v for i, j, v in off if abs(i - j) == 1)
            print(f"   {m:<2} {level:<8} errors {n_err:>4} | adjacent (|pred−true| = 1) {adj} = {adj / max(n_err, 1):.3f} | "
                  f"distant (>= 2) {n_err - adj} = {(n_err - adj) / max(n_err, 1):.3f}")
        else:
            r = table_row(m, level)
            print(f"   {m:<2} {level:<8} adjacent share {float(r.adjacent_error_share):.3f} (from v29's table; raw counts need .npz)")

## 4. Per-day (per-fold) breakdown — the day-out generalization result itself
Each fold tests on one recording day the model never saw. Macro-F1 and balanced accuracy are given per day for F1
and D, at window and mission level.
- **Bars:** the seed-averaged-F1 method, from the per-run CSV (always available), with each seed's value shown as a
  dot.
- **Diamonds:** the pooled-probability value, when the `.npz` files are present.

**2021-12-22 is a small fold** (10 missions, 2 per class).

In [ ]:
# ==============================================================================
# 4. PER-DAY TABLE + BAR CHART
# ==============================================================================
_nm = {d: int(V29_RUNS[(V29_RUNS.model == "F1") & (V29_RUNS.fold_day == d)].n_test_windows.iloc[0]) for d in DAYS}
PERDAY = []
for k, day in enumerate(DAYS):
    row = {"fold": k, "test day": day + (" (SMALL, 10 missions)" if day == SMALL_FOLD_DAY else ""), "test windows": _nm[day]}
    for m in ("F1", "D"):
        d = V29_RUNS[(V29_RUNS.model == m) & (V29_RUNS.fold == k)]
        row[f"{m} win macro-F1"] = d.win_macro_f1.mean()
        row[f"{m} win bal-acc"] = d.win_balanced_accuracy.mean()
        row[f"{m} mission macro-F1"] = d.mis_macro_f1.mean()
        if m in POOL:
            row[f"{m} win macro-F1 (pooled-prob)"] = fold_pooled(m, "window")[k]
    PERDAY.append(row)
PERDAY = pd.DataFrame(PERDAY).set_index("fold")
print("Per unseen day — seed-averaged-F1 method (mean of the 2 seeds' own per-fold scores)"
      + ("; pooled-probability columns included" if "F1" in POOL else "") + "\n")
print(md_table(PERDAY))
PERDAY.to_csv(WORK / "v31_per_day.csv")

fig, axs = plt.subplots(1, 3, figsize=(17, 4.8), dpi=150)
x = np.arange(len(DAYS))
w = 0.36
for ax, (col, lab) in zip(axs, (("win_macro_f1", "window macro-F1"), ("win_balanced_accuracy", "window balanced accuracy"),
                                ("mis_macro_f1", "mission macro-F1"))):
    for off, m in ((-w / 2, "F1"), (w / 2, "D")):
        d = V29_RUNS[V29_RUNS.model == m]
        vals = [d[d.fold == k][col].values for k in range(len(DAYS))]
        ax.bar(x + off, [v.mean() for v in vals], w, color=MODEL_COLORS[m], edgecolor="black", linewidth=0.5,
               label=f"{m}{' (headline)' if m == 'F1' else ' (comparison)'}")
        for k, v in enumerate(vals):
            ax.scatter([x[k] + off] * len(v), v, s=12, color="black", zorder=3)
        if m in POOL and col != "win_balanced_accuracy":
            pv = fold_pooled(m, "window" if col.startswith("win") else "mission")
            ax.scatter(x + off, pv, marker="D", s=26, facecolor="white", edgecolor="black", zorder=4,
                       label="pooled-probability" if m == "F1" else None)
    ax.set_xticks(x)
    ax.set_xticklabels([d + ("\n(small, 10 missions)" if d == SMALL_FOLD_DAY else "") for d in DAYS], fontsize=8)
    ax.set_ylim(0.5, 1.02)
    ax.set_ylabel(lab)
    ax.set_title(f"{lab} per unseen day", fontsize=10, fontweight="bold")
axs[0].legend(fontsize=8, loc="lower left")
fig.tight_layout()
save_fig(fig, "f4_per_day_F1_vs_D", f"{PROTOCOL_TAG}. Bars = mean of the 2 seeds' per-fold scores (dots = each seed); "
                                    f"diamonds = pooled-probability score where .npz files are available. {tie_note()}")

## 5. Safety-relevant breakdown
Macro-F1 blends two different jobs into one number:
- **detecting that a fault exists** at all;
- **grading its severity** (1–4 broken propellers).

For an operator these errors are not equal:
- a **missed fault** (a faulty flight called Normal) is the dangerous error;
- a **false alarm** (a Normal flight called faulty) costs an unnecessary inspection;
- a **severity mix-up between two faulty classes** still triggers an inspection.

This section separates the first two from severity grading, at window and mission level.

In [ ]:
# ==============================================================================
# 5. SAFETY: missed faults and false alarms (raw counts need .npz; shares from v29's table otherwise)
# ==============================================================================
SAFE = []
for m in ("F1", "D"):
    for level in ("window", "mission"):
        r = table_row(m, level)
        row = {"model": m, "level": level, "missed-fault share": float(r.faulty_pred_normal),
               "false-alarm share": float(r.normal_pred_faulty), "adjacent-error share": float(r.adjacent_error_share)}
        if m in POOL:
            p = POOL[m]
            y, pred = (p["meta"].cls.values, p["P"].argmax(1)) if level == "window" else (p["m_y"], p["m_P"].argmax(1))
            row["missed faults (n / faulty)"] = f"{int(((y > 0) & (pred == 0)).sum())} / {int((y > 0).sum())}"
            row["false alarms (n / normal)"] = f"{int(((y == 0) & (pred > 0)).sum())} / {int((y == 0).sum())}"
            row["severity errors among faulty (n)"] = int(((y > 0) & (pred > 0) & (pred != y)).sum())
        else:
            row["missed faults (n / faulty)"] = row["false alarms (n / normal)"] = "needs .npz"
        SAFE.append(row)
SAFE = pd.DataFrame(SAFE).set_index(["model", "level"])
print(md_table(SAFE.reset_index().set_index("model")))
SAFE.to_csv(WORK / "v31_safety.csv")
fig, axs = plt.subplots(1, 2, figsize=(11, 4.2), dpi=150)
for ax, level in zip(axs, ("window", "mission")):
    vals = {m: (SAFE.loc[(m, level), "missed-fault share"], SAFE.loc[(m, level), "false-alarm share"]) for m in ("F1", "D")}
    xx = np.arange(2)
    for off, m in ((-0.18, "F1"), (0.18, "D")):
        b = ax.bar(xx + off, vals[m], 0.36, color=MODEL_COLORS[m], edgecolor="black", linewidth=0.5, label=m)
        ax.bar_label(b, labels=[f"{v:.3f}" for v in vals[m]], fontsize=8, padding=2)
    ax.set_xticks(xx)
    ax.set_xticklabels(["missed fault\n(faulty -> Normal)", "false alarm\n(Normal -> faulty)"])
    ax.set_ylabel("share")
    ax.set_ylim(0, max(0.05, max(max(v) for v in vals.values()) * 1.4))
    ax.set_title(f"{level} level", fontsize=10, fontweight="bold")
    ax.legend(fontsize=8)
fig.suptitle("Safety-relevant errors — F1 vs D", fontsize=11, fontweight="bold")
fig.tight_layout()
save_fig(fig, "f5_safety_F1_vs_D", f"{PROTOCOL_TAG}. Shares from v29's pooled table (computed from the per-window "
                                   f"predictions); raw counts are in the printed table. {tie_note()}")

## 6. Per-class precision / recall / F1 — F1
Grouped bars in the style of the project's original bi-modal per-class figure: Precision `#4c72b0`, Recall
`#55a868`, F1-Score `#c44e52`, black edges and a 0–1 axis.

In [ ]:
# ==============================================================================
# 6. PER-CLASS BAR CHART (F1; needs .npz for precision and F1)
# ==============================================================================
for level in ("window", "mission"):
    name = f"f6_per_class_F1_{level}"
    if "F1" not in POOL:
        skip_fig(name, MISSING["F1"])
        continue
    t = PERCLASS[("F1", level)]
    x = np.arange(N_CLASSES)
    width = 0.25
    fig = plt.figure(figsize=(8, 5), dpi=150)
    for off, (col, lab) in zip((-width, 0, width), (("precision", "Precision"), ("recall", "Recall"), ("F1", "F1-Score"))):
        plt.bar(x + off, t[col].astype(float).values, width, label=lab, color=PRF_COLORS[lab], edgecolor="black", linewidth=0.5)
    plt.xlabel("Fault Severity (Broken Propeller Count)", fontsize=11)
    plt.ylabel("Score", fontsize=11)
    plt.title(f"F1 — Per-Class Diagnostic Performance ({level} level)", fontsize=12, fontweight="bold", pad=12)
    plt.xticks(x, CLASS_NAMES, fontsize=10)
    plt.ylim([0.0, 1.0])
    plt.legend(loc="lower right", fontsize=10)
    plt.tight_layout()
    save_fig(fig, name, f"{PROTOCOL_TAG}; support per class = {t['support'].tolist()} {level}s.")

## 7. Gate diagnostics — F1's distinguishing mechanism
F1 computes a content-adaptive weighting per window: `[g_a, g_i] = softmax(MLP([audio summary ; IMU summary]))`.
**g_a** is the weight on the audio token block, and **g_i = 1 − g_a** is the weight on the IMU block.

**(a) Distribution.** g_a over all test windows, pooled over folds and both seeds (needs F1's `.npz` files).

**(b) By day.** g_a's mean and spread per unseen day and per seed, from the `.npz` files, or from the per-run gate
statistics v29 logged at test time if the files are missing.

**Hypothesis test.** If the gate tracks modality reliability, it should shift toward IMU on days where IMU is stronger
and toward audio on days where audio is stronger. Two checks are computed across the 5 folds, with Pearson and
Spearman correlations and n = 5:
1. fold mean g_a vs the fold's **F1 − I** macro-F1 gap (as requested);
2. fold mean g_a vs the fold's **A − I** gap, which is the most direct "audio stronger that day" measure. The
   hypothesis predicts a **positive** correlation here.

Because the per-fold mean gate differs between the two seeds of the same fold, the between-fold spread is also
compared with the seed-to-seed spread within folds. If seeds disagree more than days do, a fold's gate value is not a
stable property of that day.

This is a diagnostic. The result is reported as it comes out.

In [ ]:
# ==============================================================================
# 7. GATE DIAGNOSTICS (F1)
# ==============================================================================
F1R = V29_RUNS[V29_RUNS.model == "F1"].sort_values(["fold", "seed"])
GA_FOLD_SEED = F1R.pivot(index="fold", columns="seed", values="g_a_mean")          # per-run test-set mean g_a (logged by v29)
GA_FOLD = GA_FOLD_SEED.mean(axis=1).values                                           # == mean over both seeds' windows

# (a) pooled distribution — needs the per-window g_a
if "F1" in POOL and "g_a" in POOL["F1"]["per_seed"][0]:
    ga_all = np.concatenate([ps["g_a"] for ps in POOL["F1"]["per_seed"]])
    days_all = np.concatenate([ps["day"] for ps in POOL["F1"]["per_seed"]])
    seeds_all = np.concatenate([[s] * len(ps["g_a"]) for s, ps in zip(SEEDS, POOL["F1"]["per_seed"])])
    cls_all = np.concatenate([ps["cls"] for ps in POOL["F1"]["per_seed"]])
    fig, ax = plt.subplots(figsize=(8, 4.2), dpi=150)
    for s, col in zip(SEEDS, ("#4c72b0", "#dd8452")):
        ax.hist(ga_all[seeds_all == s], bins=40, range=(0, 1), alpha=0.6, color=col, edgecolor="white", label=f"seed {s}")
    ax.axvline(0.5, color="black", lw=1, ls="--")
    ax.set_xlabel("g_a (weight on the audio token block; g_i = 1 - g_a)")
    ax.set_ylabel("test windows")
    ax.set_title("F1 content gate: distribution of g_a over all test windows", fontsize=11, fontweight="bold")
    ax.legend()
    fig.tight_layout()
    save_fig(fig, "f7a_gate_distribution_F1", f"{PROTOCOL_TAG}; {len(ga_all)} window-seed values "
                                              f"(5238 windows x 2 seeds). Left of 0.5 = leaning on IMU.")
    print(f"g_a pooled: mean {ga_all.mean():.3f}, sd {ga_all.std():.3f}, share < 0.5 (IMU-leaning) {(ga_all < 0.5).mean():.3f}")
    print("mean g_a by TRUE class: " + ", ".join(f"class {c} {ga_all[cls_all == c].mean():.3f}" for c in range(N_CLASSES)))
else:
    skip_fig("f7a_gate_distribution_F1", MISSING["F1"] or ["g_a array not present in F1 .npz"])
    ga_all = None

# (b) by day, per seed
fig, ax = plt.subplots(figsize=(9, 4.4), dpi=150)
x = np.arange(len(DAYS))
if ga_all is not None:
    data = [ga_all[(days_all == d) & (seeds_all == s)] for d in DAYS for s in SEEDS]
    pos = [k + off for k in x for off in (-0.18, 0.18)]
    bp = ax.boxplot(data, positions=pos, widths=0.3, showfliers=False, patch_artist=True)
    for i, b in enumerate(bp["boxes"]):
        b.set_facecolor("#4c72b0" if i % 2 == 0 else "#dd8452")
        b.set_alpha(0.6)
    src = "per-window g_a from F1's .npz files (boxes: IQR; whiskers 1.5 IQR)"
else:
    for off, s, col in ((-0.18, 42, "#4c72b0"), (0.18, 123, "#dd8452")):
        r = F1R[F1R.seed == s].set_index("fold").loc[range(len(DAYS))]
        ax.errorbar(x + off, r.g_a_mean, yerr=r.g_a_std, fmt="o", color=col, capsize=4, label=f"seed {s}: mean ± sd")
        ax.scatter(x + off, r.g_a_min, marker="v", color=col, s=14)
        ax.scatter(x + off, r.g_a_max, marker="^", color=col, s=14)
    src = "per-run test-set g_a mean ± sd, min/max (logged by v29 at test time; .npz files missing)"
ax.plot(x, GA_FOLD, "k_", markersize=22, mew=2, label="fold mean (both seeds)")
ax.axhline(0.5, color="black", lw=0.8, ls="--")
ax.set_xticks(x)
ax.set_xticklabels([d + ("\n(small)" if d == SMALL_FOLD_DAY else "") for d in DAYS], fontsize=8)
ax.set_ylabel("g_a (audio weight)")
ax.set_ylim(0, 1)
ax.set_title("F1 gate by unseen day and seed (blue = seed 42, orange = seed 123)", fontsize=11, fontweight="bold")
ax.legend(fontsize=8, loc="upper right")
fig.tight_layout()
save_fig(fig, "f7b_gate_by_day_F1", f"{PROTOCOL_TAG}. Source: {src}.")

# ---- hypothesis test across the 5 folds ----
GAP = {"F1 − I (seed-averaged-F1)": fold_seedavg("F1", "win_macro_f1") - fold_seedavg("I", "win_macro_f1"),
       "A − I (seed-averaged-F1)": fold_seedavg("A", "win_macro_f1") - fold_seedavg("I", "win_macro_f1")}
if all(m in POOL for m in ("F1", "I")):
    GAP["F1 − I (pooled-probability)"] = fold_pooled("F1", "window") - fold_pooled("I", "window")
if all(m in POOL for m in ("A", "I")):
    GAP["A − I (pooled-probability)"] = fold_pooled("A", "window") - fold_pooled("I", "window")
print("Per fold: mean g_a (both seeds) and the performance gaps (window macro-F1):")
print(md_table(pd.DataFrame({"test day": DAYS, "mean g_a": GA_FOLD, "g_a seed 42": GA_FOLD_SEED[42].values,
                             "g_a seed 123": GA_FOLD_SEED[123].values, **GAP}).set_index("test day"), "{:+.3f}"))
GATE_CORR = {}
for name, gap in GAP.items():
    r_p, p_p = stats.pearsonr(GA_FOLD, gap)
    r_s, p_s = stats.spearmanr(GA_FOLD, gap)
    GATE_CORR[name] = (r_p, p_p, r_s, p_s)
    print(f"corr(fold mean g_a, {name}): Pearson {r_p:+.3f} (p {p_p:.2f}) | Spearman {r_s:+.3f} (p {p_s:.2f}) | n = 5 folds")
_between = float(np.std(GA_FOLD, ddof=1))
_within = float(np.mean(np.abs(GA_FOLD_SEED[42].values - GA_FOLD_SEED[123].values)) / math.sqrt(2))
_per_seed_ai = {s: stats.spearmanr(GA_FOLD_SEED[s].values, GAP["A − I (seed-averaged-F1)"])[0] for s in SEEDS}
print(f"Stability: SD of the 5 fold means {_between:.3f} vs typical seed-to-seed SD within a fold {_within:.3f}; "
      f"per-seed Spearman(g_a, A − I): " + ", ".join(f"seed {s} {v:+.2f}" for s, v in _per_seed_ai.items()))
_rai = GATE_CORR["A − I (seed-averaged-F1)"]
GATE_VERDICT = (
    ("SUPPORTED: " if _rai[2] > 0.7 and _rai[0] > 0.7 else "NOT SUPPORTED: ")
    + f"across the 5 unseen days, F1's mean audio weight does "
    + ("" if _rai[2] > 0.7 and _rai[0] > 0.7 else "not ")
    + f"track which modality was stronger that day (Spearman {_rai[2]:+.2f}, Pearson {_rai[0]:+.2f} with the A − I gap; "
      f"with the F1 − I gap: Spearman {GATE_CORR['F1 − I (seed-averaged-F1)'][2]:+.2f}, Pearson {GATE_CORR['F1 − I (seed-averaged-F1)'][0]:+.2f}). "
    + (f"The fold-level gate is also unstable across seeds (seed-to-seed SD {_within:.2f} vs between-day SD {_between:.2f}), "
       f"so a day's mean gate value is not a reproducible property of that day." if _within >= _between else
       f"The fold-level gate is stable across seeds (seed-to-seed SD {_within:.2f} < between-day SD {_between:.2f}).")
    + " With n = 5 folds, even a strong correlation would be weak evidence.")
print("\nGATE DIAGNOSTIC RESULT: " + GATE_VERDICT)

## 8. Complementarity — audio alone vs IMU alone vs F1, at mission level
Each of the 95 missions is classified by A (audio-only) and I (IMU-only), both from v29's saved day-out predictions
on the identical paired windows. The two predictions are then compared:
- both correct;
- audio-only correct;
- IMU-only correct;
- both wrong.

**"Oracle either-correct"** is the ceiling a perfect modality selector could reach. It is shown next to A alone,
I alone and F1 (with D for comparison).

This needs A's, I's and F1's per-window files. The mission set is asserted identical across A, I and F1, and equal
to the hash-verified 95-mission set from section 1.

In [ ]:
# ==============================================================================
# 8. COMPLEMENTARITY FIGURE (needs A, I, F1 .npz)
# ==============================================================================
_need = [m for m in ("A", "I", "F1") if m not in POOL]
if _need:
    skip_fig("f8_complementarity", sum((MISSING[m] for m in _need), []))
    COMP = None
else:
    mp = {m: pd.DataFrame({"mission": POOL[m]["m_names"], "true": POOL[m]["m_y"], "pred": POOL[m]["m_P"].argmax(1)}).set_index("mission")
          for m in ("A", "I", "F1") + (("D",) if "D" in POOL else ())}
    same = all(mp[m].index.equals(mp["A"].index) and (mp[m].true == mp["A"].true).all() for m in mp)
    hook("windows", same and len(mp["A"]) == EXPECTED_N_MISSIONS,
         f"A, I, F1{', D' if 'D' in mp else ''} mission-level predictions are paired on the identical {len(mp['A'])} missions "
         f"(same ids, same true classes)" + (" — the hash-verified set from section 1" if VERIFY_WINDOW_HASH else ""))
    a_ok, i_ok, f_ok = [(mp[m].pred == mp[m].true).values for m in ("A", "I", "F1")]
    COMP = {"both correct": int((a_ok & i_ok).sum()), "audio-only correct": int((a_ok & ~i_ok).sum()),
            "IMU-only correct": int((~a_ok & i_ok).sum()), "both wrong": int((~a_ok & ~i_ok).sum())}
    n = len(a_ok)
    oracle = (a_ok | i_ok).mean()
    print("Mission-level complementarity (95 missions): " + ", ".join(f"{k} {v} ({v / n:.3f})" for k, v in COMP.items()))
    print(f"Oracle either-correct {oracle:.3f} | A {a_ok.mean():.3f} | I {i_ok.mean():.3f} | F1 {f_ok.mean():.3f}"
          + (f" | D {(mp['D'].pred == mp['D'].true).mean():.3f}" if "D" in mp else ""))
    print(f"F1 on the missions where the modalities disagree: audio-only-correct ones {f_ok[a_ok & ~i_ok].mean() if (a_ok & ~i_ok).any() else float('nan'):.3f}, "
          f"IMU-only-correct ones {f_ok[~a_ok & i_ok].mean() if (~a_ok & i_ok).any() else float('nan'):.3f}, both-wrong ones "
          f"{f_ok[~a_ok & ~i_ok].mean() if (~a_ok & ~i_ok).any() else float('nan'):.3f}")
    fig, axs = plt.subplots(1, 2, figsize=(13, 4.8), dpi=150, gridspec_kw={"width_ratios": [1, 1.3]})
    grid = np.array([[COMP["both correct"], COMP["IMU-only correct"]], [COMP["audio-only correct"], COMP["both wrong"]]])
    axs[0].imshow(grid, cmap="Blues", vmin=0)
    axs[0].grid(False)
    for i in range(2):
        for j in range(2):
            axs[0].text(j, i, f"{grid[i, j]}\n({grid[i, j] / n:.1%})", ha="center", va="center", fontsize=11,
                        color="white" if grid[i, j] > grid.max() / 2 else "black")
    axs[0].set_xticks([0, 1])
    axs[0].set_xticklabels(["audio correct", "audio wrong"])
    axs[0].set_yticks([0, 1])
    axs[0].set_yticklabels(["IMU correct", "IMU wrong"])
    axs[0].set_title("Audio-only vs IMU-only, per mission", fontsize=11, fontweight="bold")
    names = ["A (audio)", "I (IMU)", "F1 (headline)"] + (["D (comparison)"] if "D" in mp else []) + ["oracle\neither-correct"]
    vals = [a_ok.mean(), i_ok.mean(), f_ok.mean()] + ([(mp["D"].pred == mp["D"].true).mean()] if "D" in mp else []) + [oracle]
    cols = [MODEL_COLORS["A"], MODEL_COLORS["I"], MODEL_COLORS["F1"]] + ([MODEL_COLORS["D"]] if "D" in mp else []) + [MODEL_COLORS["oracle"]]
    b = axs[1].bar(names, vals, color=cols, edgecolor="black", linewidth=0.5)
    axs[1].bar_label(b, labels=[f"{v:.3f}" for v in vals], fontsize=9, padding=2)
    axs[1].set_ylim(0.5, 1.05)
    axs[1].set_ylabel("mission-level accuracy")
    axs[1].set_title("Mission accuracy vs the complementarity ceiling", fontsize=11, fontweight="bold")
    fig.tight_layout()
    save_fig(fig, "f8_complementarity", f"{PROTOCOL_TAG}; 95 missions, mission prediction = argmax of the mean window "
                                        f"probability. Oracle = correct if A or I is correct (not achievable without "
                                        f"knowing the answer). {tie_note()}")

## 9. Appendix — all 8 models (context, not the headline result)
The rows are loaded unchanged from v30's combined table (`v30_results_table.csv`), which has v29's six rows plus CA
and CAG under the identical protocol. If v30's table is missing, v29's six rows are shown and the gap is stated.

In [ ]:
# ==============================================================================
# 9. APPENDIX TABLE
# ==============================================================================
APP = V30_TABLE.copy() if V30_TABLE is not None else V29_TABLE.copy()
if V30_TABLE is None:
    print("APPENDIX NOTE: v30_results_table.csv not found — CA and CAG are NOT shown. Searched: "
          + ", ".join(p.replace("{repo}", str(PROJECT_REPO)) for p in SEARCH["v30_table"]))
for level in ("window", "mission"):
    t = APP[APP.level == level].set_index("model").reindex([m for m in ALL_MODELS if m in set(APP.model)])
    t["macro-F1 [95% CI]"] = [f"{a:.3f} [{b:.3f}, {c:.3f}]" for a, b, c in zip(t.macro_f1, t.f1_ci_low, t.f1_ci_high)]
    out = t[["params", "accuracy", "macro-F1 [95% CI]", "balanced_accuracy", "macro_auroc", "faulty_pred_normal", "normal_pred_faulty"]]
    print(f"\nAPPENDIX — all models, pooled day-out, {level} level ({PROTOCOL_TAG})\n" + md_table(out))
    out.to_csv(WORK / f"v31_appendix_{level}.csv")
    render_table_png(out, f"tA_appendix_{level}", f"Appendix — all models, {level} level (context; not the headline)",
                     f"{PROTOCOL_TAG}. Identical protocol and metrics for every row (v29 + v30).", col_width=1.35)

## 10. Limitations (generated from the numbers above)

In [ ]:
# ==============================================================================
# 10. LIMITATIONS + figure list + final hooks (incl. "nothing was trained")
# ==============================================================================
_f, _d, _i = table_row("F1", "mission"), table_row("D", "mission"), table_row("I", "mission")
_fw, _iw = table_row("F1", "window"), table_row("I", "window")
_seeds_per_model = V29_RUNS.groupby("model").seed.nunique().to_dict()
_res_fd, _flag_fd = paired_both("F1", "D", "window")
_sf = _res_fd["seed-averaged-F1"]
LIM = [
    "LIMITATIONS",
    f"- Statistical power: {len(DAYS)} day-out folds and {sorted(set(_seeds_per_model.values()))} seed(s) per model -> paired "
    f"fold tests have df = {len(DAYS) - 1}; a difference must be large and consistent across all days to be detectable.",
    f"- F1 vs D is a statistical tie: mission macro-F1 {_f.macro_f1:.3f} [{_f.f1_ci_low:.3f}, {_f.f1_ci_high:.3f}] vs "
    f"{_d.macro_f1:.3f} [{_d.f1_ci_low:.3f}, {_d.f1_ci_high:.3f}]; per-fold window F1 − D (seed-averaged-F1 method) mean "
    f"{_sf['mean']:+.4f}, F1 wins {_sf['wins']}/5, p {_sf['p']:.2f} [{_flag_fd}]. F1 is the headline for its interpretable, "
    f"content-adaptive fusion, not for higher accuracy.",
    f"- F1 does NOT beat IMU alone: mission macro-F1 F1 {_f.macro_f1:.3f} [{_f.f1_ci_low:.3f}, {_f.f1_ci_high:.3f}] vs "
    f"I {_i.macro_f1:.3f} [{_i.f1_ci_low:.3f}, {_i.f1_ci_high:.3f}]; window {_fw.macro_f1:.3f} vs {_iw.macro_f1:.3f}.",
    f"- Gate diagnostic: {GATE_VERDICT}",
    f"- These are leave-one-day-out results on {EXPECTED_N_MISSIONS} missions, not a held-out, never-touched final test set: "
    f"many architectures were compared on these same folds over the project, so the best-looking numbers carry some "
    f"selection optimism.",
    "- Standard-split numbers reported earlier in the project (D and F1's predecessor architectures at ~0.89-0.91 macro-F1) "
    "used a random mission-level split whose test days also appear in training; they are NOT comparable to these day-out "
    "numbers and must not be cited alongside them without this caveat.",
]
LIMITATIONS = "\n".join(LIM)
print(LIMITATIONS)
(WORK / "v31_limitations.txt").write_text(LIMITATIONS, encoding="utf-8")

print("\nFIGURES AND TABLES:")
for name, cap in FIGURES:
    print(f"   {name:<32} " + (f"SKIPPED ({cap.split('— ', 1)[-1]})" if cap.startswith("SKIPPED") else f"{FIG_DIR / (name + '.png')} (+ .pdf)"))
print(f"   CSV tables: {sorted(p.name for p in WORK.glob('v31_*.csv'))}")

hook("no-training", "torch" not in sys.modules,
     "torch was never imported: no model was built, loaded, trained or run in this notebook", stop=False)
hook("captions", all(("fold" in c and "seed" in c) or c.startswith("SKIPPED") for _, c in FIGURES),
     "every figure states its protocol (day-out), fold count and seed count; skipped figures name their missing files", stop=False)
print("\n" + "=" * 90 + "\nVERIFICATION HOOKS\n" + "=" * 90)
for name, res in HOOKS.items():
    print(f"  {name:<14} {'PASS' if all(ok for ok, _ in res) else 'FAIL'}")
    for ok, msg in res:
        print(f"      {'PASS' if ok else 'FAIL'}: {msg}")